In [1]:
import contextlib
from datetime import datetime
from pathlib import Path
from uuid import uuid4

import plotly.figure_factory as ff
import plotly.graph_objects as go
import torch
import torch.nn.functional as F
import wandb
from tqdm.autonotebook import tqdm

from src.chart_transport.constraint import (
    LagrangianConstraintConfig,
    LossConstraintConfig,
)
from src.data.gaussian_mixture.data import MultimodalGaussianDataConfig
from src.experiments.multimodal_gaussian.canonical import (
    get_canonical_chart_transport_configs,
    get_canonical_chart_transport_monitor_configs,
)
from src.experiments.multimodal_gaussian.config import MultimodalGaussianTrainingConfig
from src.experiments.multimodal_gaussian.state import MultimodalTrainingRuntime

/tmp/ipykernel_2140013/587479404.py:11: TqdmExperimentalWarning:

Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)



In [2]:
num_modes = 8
latent_dimension = 2
ambient_dimension = 2
experiment_name = f"{datetime.now():%m%d%H%M%S}-{uuid4().hex[:6]}"
wandb_project = "diffusive-latent-generation"

data_config = MultimodalGaussianDataConfig.initialize(
    num_modes=num_modes,
    mode_std=0.1,
    offset=0.0,
    ambient_dimension=ambient_dimension,
)

training_config = MultimodalGaussianTrainingConfig.initialize(
    seed=0,
    train_batch_size=4096,
    eval_batch_size=4096,
    integrated_n_steps=1_000_000,
    chart_transport_config=get_canonical_chart_transport_configs(
        data_config=data_config,
        latent_dimension=latent_dimension,
    ),
    monitor_config=get_canonical_chart_transport_monitor_configs(),
    folder=(
        Path("/home/nlyu/Code/diffusive-latent-generation/artifacts/multimodal_gaussian")
        / experiment_name
    ),
)

training_config.visualize()

In [3]:
cuda_device = 1
runtime = MultimodalTrainingRuntime.initialize(
    tc=training_config,
    cuda_device=cuda_device,
    wandb_project=wandb_project,
    run_name=experiment_name,
)

Using bfloat16 Automatic Mixed Precision (AMP)


Seed set to 0
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/nlyu/.netrc.
wandb: Currently logged in as: lyuxingjian (lyuxingjian-na) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Pretrain

Chart pretrain is now delegated to the runtime. It logs constraint and conditioning monitors on the configured schedule.


In [4]:
latest_pretrain_metrics = runtime.chart_pretrain()
latest_pretrain_metrics

pretrain:   0%|          | 0/1000 [00:00<?, ?it/s]

[pretrain] step 1/1000: train: chart_loss=1.0077, data_cycle_loss=0.4342, prior_cycle_loss=0.5734, zero_mean_loss=0.0488, latent_norm_loss=0.1513; monitor: constraint_reconstruction_mean=4.4165, constraint_reconstruction_max=6.4266, constraint_latent_norm_mean=5.5531, encoder_conditioning_mean=5.2282, encoder_conditioning_max=551.3914, decoder_conditioning_mean=0.8978, decoder_conditioning_max=246.0355
[pretrain] step 1000/1000: train: chart_loss=0.0001, data_cycle_loss=0.0000, prior_cycle_loss=0.0000, zero_mean_loss=0.1231, latent_norm_loss=1.0648; monitor: constraint_reconstruction_mean=0.0061, constraint_reconstruction_max=0.0243, constraint_latent_norm_mean=1.2653, encoder_conditioning_mean=1.1018, encoder_conditioning_max=4.9686, decoder_conditioning_mean=1.0519, decoder_conditioning_max=2.8634


{'train_chart_loss': 0.0001397110754624009,
 'train_data_cycle_loss': 9.42230508371722e-06,
 'train_prior_cycle_loss': 2.380467049079016e-05,
 'train_zero_mean_loss': 0.12305450439453125,
 'train_latent_norm_loss': 1.0648409128189087,
 'monitor_constraint_reconstruction_mean': 0.006069295108318329,
 'monitor_constraint_reconstruction_max': 0.024346832185983658,
 'monitor_constraint_latent_norm_mean': 1.2652615308761597,
 'monitor_encoder_conditioning_mean': 1.1018071174621582,
 'monitor_encoder_conditioning_max': 4.968629837036133,
 'monitor_decoder_conditioning_mean': 1.0518932342529297,
 'monitor_decoder_conditioning_max': 2.8634419441223145}

## Train noise critic

In [5]:
latest_critic_pretrain_metrics = runtime.critic_pretrain()
latest_critic_pretrain_metrics

critic_pretrain:   0%|          | 0/1000 [00:00<?, ?it/s]

[critic_pretrain] step 1/1000: train: critic_loss=1.2567; monitor: critic_monitor_snapshot_score_norm_mean=286.2863, critic_monitor_transport_norm_mean=62.2226, critic_monitor_transport_norm_max=92.8228, critic_monitor_transport_t_min=0.0100, critic_monitor_transport_t_max=0.9900
[critic_pretrain] step 1000/1000: train: critic_loss=0.3206; monitor: critic_monitor_snapshot_score_norm_mean=14.3740, critic_monitor_transport_norm_mean=3.1245, critic_monitor_transport_norm_max=7.4572, critic_monitor_transport_t_min=0.0100, critic_monitor_transport_t_max=0.9900


{'train_critic_loss': 0.3206021189689636,
 'monitor_critic_monitor_snapshot_score_norm_mean': 14.374011039733887,
 'monitor_critic_monitor_transport_norm_mean': 3.1244945526123047,
 'monitor_critic_monitor_transport_norm_max': 7.457155227661133,
 'monitor_critic_monitor_transport_t_min': 0.009999999776482582,
 'monitor_critic_monitor_transport_t_max': 0.9900000095367432}

## Integrated training

Integrated training is now delegated to the runtime. Each integrated step performs one chart-repair update, `n_critic_updates_every_transport_step` critic updates, then one chart-transport update. The integrated monitor stack logs constraint, critic, conditioning, and sampling monitors on the configured schedule.


In [ ]:
latest_integrated_metrics = runtime.integrated()
latest_integrated_metrics

integrated:   0%|          | 0/1000000 [00:00<?, ?it/s]

[integrated] step 1/1000000: train: critic=0.3334, repair=0.0001, data_cycle=0.0000, prior_cycle=0.0001, data_dual=0.0000, prior_dual=0.0000, transport=0.0022, enc_transport=0.0012, dec_transport=0.0010, field=2.2516; monitor: recon_err=0.0158, latent_norm=1.2061, score=13.1151, field=3.2539, enc_cond=1.0928, dec_cond=1.0722, kl_0p001=4.9465, kl_0p003=4.6520, kl_0p01=4.6023, kl_0p03=4.5873
